## Data Catalog: Field Shift Crop Rotation (Andalusia)

### 1. Study region and why four zones

Andalusia is one of Spain's largest farming regions and is strongly exposed to
drought, water scarcity and soil erosion. It is also very varied: no single
average describes it, and a crop rotation that works in one area may fail in
another. For that reason the project does not treat Andalusia as one block.
Instead it uses four contrasting zones, each represented by one reference
point, and runs the same analysis on every zone.

| Zone | Reference point | Lat | Lon | Farming system | Why it was chosen |
|---|---|---|---|---|---|
| 1. Campiña | near Écija | 37.54 | -5.08 | Rainfed cereals, sunflower | Core Guadalquivir farmland with a classic cereal-sunflower rotation, very hot dry summers and bare fields after harvest. Baseline zone. |
| 2. Olive belt | near Úbeda | 38.01 | -3.37 | Olives on slopes | Sloping land where erosion and bare soil between trees are the main issues, so cover crops are the relevant strategy. |
| 3. Highlands | near Guadix | 37.30 | -3.14 | Dry cereals, almonds | High, dry inland area with low rainfall and cold nights: shows frost risk and water scarcity. |
| 4. Western lowlands | near Jerez | 36.68 | -6.14 | Cereals, sunflower | Closer to the Atlantic with a different rainfall pattern, showing climate variation across the region. |

Together the zones cover four contrasting conditions: lowland rainfed cereals,
sloping olive land, dry highlands and an Atlantic-influenced lowland.

Limitations:
- Reference points are approximate and should be verified on satellite imagery.
- Descriptions of each zone are general agronomic knowledge, not verified
  statistics. TODO: add sources.
- Time standard: all daily data is in UTC so datasets can be joined on the
  same calendar day.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, classification_report

**FUNCTIONS:**

In [2]:
def add_data_index(df):
    df["date"] = pd.to_datetime(df["YEAR"].astype(str) + df["DOY"].astype(str).str.zfill(3), format="%Y%j")
    df = df.drop(columns=["YEAR", "DOY"]).set_index("date")
    return df

In [3]:
def to_monthly(d):
    return d.resample("MS").agg(
        t2m_mean=("T2M", "mean"),
        t2m_max_mean=("T2M_MAX", "mean"),
        t2m_max_peak=("T2M_MAX", "max"),
        t2m_min_mean=("T2M_MIN", "mean"),
        t2m_min_low=("T2M_MIN", "min"),
        precip_mm=("PRECTOTCORR", "sum"),
        rain_days=("PRECTOTCORR", lambda s: (s >= 1).sum()),
        rh_mean=("RH2M", "mean"),
        wind_mean=("WS2M", "mean"),
        soil_root_mean=("GWETROOT", "mean"),
        soil_top_mean=("GWETTOP", "mean"),
        frost_days=("T2M_MIN", lambda s: (s < 0).sum()),
        heat_days=("T2M_MAX", lambda s: (s > 35).sum()),
        valid_days=("T2M", "count"),
    ).rename_axis("month")

### 2. Datasets

#### 2.1 NASA POWER (climate baseline)

NASA/POWER Source Native Resolution Daily Data 
Dates (month/day/year): 01/01/1981 through 10/31/2026 in UTC


The value for missing source data that cannot be computed or is outside of the sources availability range: -999 
parameter(s): 

T2M                   MERRA-2 Temperature at 2 Meters (C)

T2M_MAX               MERRA-2 Temperature at 2 Meters Maximum (C) 

T2M_MIN               MERRA-2 Temperature at 2 Meters Minimum (C) 

PRECTOTCORR           MERRA-2 Precipitation Corrected (mm/day)

ALLSKY_SFC_SW_DWN     CERES SYN1deg All Sky Surface Shortwave Downward Irradiance (MJ/m^2/day)

RH2M                  MERRA-2 Relative Humidity at 2 Meters (%) 

WS2M                  MERRA-2 Wind Speed at 2 Meters (m/s)

GWETROOT              MERRA-2 Root Zone Soil Wetness (1)

GWETTOP               MERRA-2 Surface Soil Wetness (1) 


**Ècija region**

Location: latitude  37.54   longitude -5.08 

elevation from MERRA-2: Average for 0.5 x 0.625 degree lat/lon region = 262.94 meters

In [4]:
écija_power_daily = pd.read_csv('../data/raw/power_dataset/écija.csv')
écija_power_daily.head()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M,GWETROOT,GWETTOP
0,1981,1,5.60,15.35,-0.67,0.0,-999.0,57.93,1.78,0.54,0.54
1,1981,2,5.84,14.98,0.47,0.0,-999.0,73.36,2.31,0.54,0.54
2,1981,3,6.91,14.96,0.52,0.0,-999.0,62.93,1.09,0.54,0.54
3,1981,4,6.63,14.24,1.77,0.0,-999.0,64.53,0.83,0.54,0.54
4,1981,5,5.66,14.74,0.50,0.0,-999.0,78.70,2.22,0.54,0.54


In [5]:
écija_power_daily = add_data_index(écija_power_daily)
écija_power_daily = écija_power_daily.replace(-999.0, np.nan)
écija_power_daily.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 16740 entries, 1981-01-01 to 2026-10-31
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   T2M                16740 non-null  float64
 1   T2M_MAX            16740 non-null  float64
 2   T2M_MIN            16740 non-null  float64
 3   PRECTOTCORR        16740 non-null  float64
 4   ALLSKY_SFC_SW_DWN  15584 non-null  float64
 5   RH2M               16740 non-null  float64
 6   WS2M               16740 non-null  float64
 7   GWETROOT           16740 non-null  float64
 8   GWETTOP            16740 non-null  float64
dtypes: float64(9)
memory usage: 1.3 MB


In [6]:
écija_power_daily.drop_duplicates()

,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M,GWETROOT,GWETTOP
date,,,,,,,,,
1981-01-01,5.60,15.35,-0.67,0.0,NaN,57.93,1.78,0.54,0.54
1981-01-02,5.84,14.98,0.47,0.0,NaN,73.36,2.31,0.54,0.54
1981-01-03,6.91,14.96,0.52,0.0,NaN,62.93,1.09,0.54,0.54
1981-01-04,6.63,14.24,1.77,0.0,NaN,64.53,0.83,0.54,0.54
1981-01-05,5.66,14.74,0.50,0.0,NaN,78.70,2.22,0.54,0.54
...,...,...,...,...,...,...,...,...,...
2026-10-27,18.19,23.97,13.67,0.0,NaN,46.08,1.00,0.38,0.25
2026-10-28,18.63,26.99,11.01,0.0,NaN,41.94,1.01,0.38,0.25
2026-10-29,19.38,27.17,12.62,0.0,NaN,61.22,1.24,0.38,0.24


In [7]:
écija_power_daily = écija_power_daily.drop(columns=['ALLSKY_SFC_SW_DWN'])

In [29]:
écija_power_monthly = to_monthly(écija_power_daily)
écija_power_monthly.head()

,t2m_mean,t2m_max_mean,t2m_max_peak,t2m_min_mean,t2m_min_low,precip_mm,rain_days,rh_mean,wind_mean,soil_root_mean,soil_top_mean,frost_days,heat_days,valid_days
month,,,,,,,,,,,,,,
1981-01-01,7.081935,15.285806,20.43,1.785484,-4.60,0.27,0,72.781613,1.906452,0.536452,0.533226,6,0,31
1981-02-01,8.853214,16.275714,21.41,3.094643,-0.40,14.83,4,68.872500,1.924286,0.530357,0.525714,4,0,28
1981-03-01,14.174839,21.537097,28.14,8.061290,3.10,31.12,6,64.283548,1.983226,0.519677,0.512258,0,0,31
1981-04-01,13.760667,19.871333,25.65,8.009000,3.09,68.78,15,67.518667,1.890667,0.536000,0.543333,0,0,30
1981-05-01,18.290968,26.275484,32.08,10.592903,7.17,23.18,4,55.947419,2.593871,0.524516,0.498387,0,0,31


**Guadix region:**

Location: latitude  37.3   longitude -3.14 

elevation from MERRA-2: Average for 0.5 x 0.625 degree lat/lon region = 1095.62 meters


In [9]:
guadix_power_daily = pd.read_csv('../data/raw/power_dataset/guadix.csv')
guadix_power_daily.head()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M,GWETROOT,GWETTOP
0,1981,1,2.81,11.25,-1.24,0.00,-999.0,49.28,1.01,0.49,0.48
1,1981,2,5.08,12.48,0.29,0.08,-999.0,57.10,0.84,0.49,0.48
2,1981,3,5.16,13.45,0.87,0.00,-999.0,45.55,1.12,0.49,0.48
3,1981,4,3.22,12.05,-0.82,0.00,-999.0,55.97,1.39,0.49,0.48
4,1981,5,5.56,13.29,-0.42,0.00,-999.0,65.60,0.85,0.49,0.48


In [10]:
guadix_power_daily = add_data_index(guadix_power_daily)
guadix_power_daily = guadix_power_daily.replace(-999.0, np.nan)
guadix_power_daily.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 16740 entries, 1981-01-01 to 2026-10-31
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   T2M                16740 non-null  float64
 1   T2M_MAX            16740 non-null  float64
 2   T2M_MIN            16740 non-null  float64
 3   PRECTOTCORR        16740 non-null  float64
 4   ALLSKY_SFC_SW_DWN  15584 non-null  float64
 5   RH2M               16740 non-null  float64
 6   WS2M               16740 non-null  float64
 7   GWETROOT           16740 non-null  float64
 8   GWETTOP            16740 non-null  float64
dtypes: float64(9)
memory usage: 1.3 MB


In [11]:
guadix_power_daily = guadix_power_daily.drop(columns=['ALLSKY_SFC_SW_DWN'])

In [12]:
guadix_power_monthly = to_monthly(guadix_power_daily)
guadix_power_monthly.head()

,t2m_mean,t2m_max_mean,t2m_max_peak,t2m_min_mean,t2m_min_low,precip_mm,rain_days,rh_mean,wind_mean,soil_root_mean,soil_top_mean,frost_days,heat_days,valid_days
month,,,,,,,,,,,,,,
1981-01-01,3.585161,11.129032,18.18,-1.023548,-7.39,1.88,0,66.536452,1.636452,0.488065,0.478065,16,0,31
1981-02-01,4.391071,11.493214,19.70,-0.922857,-4.86,13.61,6,65.694643,2.172500,0.484286,0.475714,19,0,28
1981-03-01,10.098387,17.330323,26.46,4.584839,-0.41,29.24,4,62.375806,2.181290,0.477419,0.449677,1,0,31
1981-04-01,9.489333,15.539333,23.49,4.191667,-0.53,69.08,13,67.101000,2.087333,0.508667,0.511000,1,0,30
1981-05-01,14.214194,21.770968,30.76,6.991290,2.60,19.03,6,54.966452,2.473871,0.495806,0.475806,0,0,31


**Jerez region**

Location: latitude  36.68   longitude -6.14 

elevation from MERRA-2: Average for 0.5 x 0.625 degree lat/lon region = 35.86 meters

In [13]:
jerez_power_daily = pd.read_csv('../data/raw/power_dataset/jerez.csv')
jerez_power_daily.head()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M,GWETROOT,GWETTOP
0,1981,1,11.31,16.21,7.17,0.0,-999.0,60.38,3.29,0.6,0.58
1,1981,2,10.89,16.37,5.51,0.0,-999.0,69.59,2.74,0.6,0.58
2,1981,3,11.30,14.01,9.39,0.0,-999.0,74.27,1.30,0.6,0.58
3,1981,4,11.56,14.50,9.74,0.0,-999.0,77.78,2.86,0.6,0.58
4,1981,5,10.65,13.95,7.57,0.0,-999.0,76.57,2.68,0.6,0.58


In [14]:
jerez_power_daily = add_data_index(jerez_power_daily)
jerez_power_daily = jerez_power_daily.replace(-999.0, np.nan)
jerez_power_daily.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 16740 entries, 1981-01-01 to 2026-10-31
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   T2M                16740 non-null  float64
 1   T2M_MAX            16740 non-null  float64
 2   T2M_MIN            16740 non-null  float64
 3   PRECTOTCORR        16740 non-null  float64
 4   ALLSKY_SFC_SW_DWN  15584 non-null  float64
 5   RH2M               16740 non-null  float64
 6   WS2M               16740 non-null  float64
 7   GWETROOT           16740 non-null  float64
 8   GWETTOP            16740 non-null  float64
dtypes: float64(9)
memory usage: 1.3 MB


In [15]:
jerez_power_daily = jerez_power_daily.drop(columns=['ALLSKY_SFC_SW_DWN'])

In [16]:
jerez_power_monthly = to_monthly(jerez_power_daily)
jerez_power_monthly.head()

,t2m_mean,t2m_max_mean,t2m_max_peak,t2m_min_mean,t2m_min_low,precip_mm,rain_days,rh_mean,wind_mean,soil_root_mean,soil_top_mean,frost_days,heat_days,valid_days
month,,,,,,,,,,,,,,
1981-01-01,11.506129,15.382581,20.44,8.406129,2.99,3.61,1,75.296129,4.335484,0.590000,0.562258,0,0,31
1981-02-01,12.419286,15.549286,18.37,9.726071,6.39,6.88,3,75.120714,4.207500,0.562500,0.531071,0,0,28
1981-03-01,15.538065,18.670968,24.42,12.881935,8.75,29.16,6,75.606129,3.442903,0.538710,0.512258,0,0,31
1981-04-01,15.285333,17.717000,21.52,12.945333,9.82,86.60,13,75.198000,3.501000,0.555667,0.541333,0,0,30
1981-05-01,17.746452,20.955484,26.79,14.495484,12.40,30.37,5,71.600645,4.156452,0.571290,0.531935,0,0,31


**Úbeda region**

Location: latitude  38.01   longitude -3.37 

elevation from MERRA-2: Average for 0.5 x 0.625 degree lat/lon region = 748.39 meters


In [17]:
úbeda_power_daily = pd.read_csv('../data/raw/power_dataset/úbeda.csv')
úbeda_power_daily.head()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M,GWETROOT,GWETTOP
0,1981,1,3.91,13.06,-0.46,0.00,-999.0,53.85,1.00,0.49,0.48
1,1981,2,4.12,12.47,-2.37,0.32,-999.0,71.25,1.00,0.49,0.48
2,1981,3,6.11,14.77,1.47,0.00,-999.0,48.80,0.99,0.49,0.48
3,1981,4,4.17,12.10,-0.16,0.00,-999.0,63.05,0.91,0.49,0.48
4,1981,5,4.94,14.32,-0.25,0.00,-999.0,74.69,1.23,0.49,0.48


In [18]:
úbeda_power_daily = add_data_index(úbeda_power_daily)
úbeda_power_daily = úbeda_power_daily.replace(-999.0, np.nan)
úbeda_power_daily.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 16740 entries, 1981-01-01 to 2026-10-31
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   T2M                16740 non-null  float64
 1   T2M_MAX            16740 non-null  float64
 2   T2M_MIN            16740 non-null  float64
 3   PRECTOTCORR        16740 non-null  float64
 4   ALLSKY_SFC_SW_DWN  15584 non-null  float64
 5   RH2M               16740 non-null  float64
 6   WS2M               16740 non-null  float64
 7   GWETROOT           16740 non-null  float64
 8   GWETTOP            16740 non-null  float64
dtypes: float64(9)
memory usage: 1.3 MB


In [19]:
úbeda_power_daily = úbeda_power_daily.drop(columns=['ALLSKY_SFC_SW_DWN'])

In [20]:
úbeda_power_monthly = to_monthly(úbeda_power_daily)
úbeda_power_monthly.head()

,t2m_mean,t2m_max_mean,t2m_max_peak,t2m_min_mean,t2m_min_low,precip_mm,rain_days,rh_mean,wind_mean,soil_root_mean,soil_top_mean,frost_days,heat_days,valid_days
month,,,,,,,,,,,,,,
1981-01-01,4.511935,12.697097,20.23,-0.600968,-6.02,2.26,0,69.194839,1.570323,0.488710,0.478710,16,0,31
1981-02-01,6.055714,13.498571,22.05,0.429643,-4.05,14.92,7,64.426071,2.030714,0.484643,0.476429,12,0,28
1981-03-01,11.657097,18.691935,26.18,6.082903,0.68,41.04,5,62.416129,2.001935,0.478710,0.459677,0,0,31
1981-04-01,11.364333,17.279333,24.90,5.848667,0.40,83.28,14,66.255667,1.932667,0.517333,0.524000,0,0,30
1981-05-01,15.842903,23.271613,32.41,8.598387,3.66,24.68,6,55.408387,2.389032,0.507419,0.494194,0,0,31


Save new datasets to the processed folder 

In [23]:
folder_path_to_daily_processed_data = '../data/processed/power_dataset/daily'
folder_path_to_monthly_processed_data = '../data/processed/power_dataset/monthly'

os.makedirs(folder_path_to_daily_processed_data, exist_ok=True)
os.makedirs(folder_path_to_monthly_processed_data, exist_ok=True)

écija_power_daily.to_csv(os.path.join(folder_path_to_daily_processed_data, 'écija_daily.csv'))
écija_power_monthly.to_csv(os.path.join(folder_path_to_monthly_processed_data, 'écija_monthly.csv'))

guadix_power_daily.to_csv(os.path.join(folder_path_to_daily_processed_data, 'guadix_daily.csv'))
guadix_power_monthly.to_csv(os.path.join(folder_path_to_monthly_processed_data, 'guadix_monthly.csv'))

jerez_power_daily.to_csv(os.path.join(folder_path_to_daily_processed_data, 'jerez_daily.csv'))
jerez_power_monthly.to_csv(os.path.join(folder_path_to_monthly_processed_data, 'jerez_monthly.csv'))

úbeda_power_daily.to_csv(os.path.join(folder_path_to_daily_processed_data, 'úbeda_daily.csv'))
úbeda_power_monthly.to_csv(os.path.join(folder_path_to_monthly_processed_data, 'úbeda_monthly.csv'))

In [27]:
écija_power_daily.head(-5)

,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,WS2M,GWETROOT,GWETTOP
date,,,,,,,,
1981-01-01,5.60,15.35,-0.67,0.0,57.93,1.78,0.54,0.54
1981-01-02,5.84,14.98,0.47,0.0,73.36,2.31,0.54,0.54
1981-01-03,6.91,14.96,0.52,0.0,62.93,1.09,0.54,0.54
1981-01-04,6.63,14.24,1.77,0.0,64.53,0.83,0.54,0.54
1981-01-05,5.66,14.74,0.50,0.0,78.70,2.22,0.54,0.54
...,...,...,...,...,...,...,...,...
2026-10-22,17.19,24.67,11.10,0.0,46.35,1.09,0.39,0.32
2026-10-23,16.57,23.68,11.66,0.0,52.71,1.49,0.39,0.30
2026-10-24,17.52,25.33,9.50,0.0,40.87,1.00,0.38,0.29
